#***Textual Data Preprocessing***


In [ ]:
text_data = Netflix_df.copy()

In [ ]:
text_data.dropna(
    subset=['date_added'],
    inplace=True
)

In [ ]:
text_data.shape

(7777, 12)

In [ ]:
import re
import string
import unicodedata

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
nlp_columns = ['title', 'director', 'cast', 'description']
text_preprocessingdf = text_data[nlp_columns]

In [ ]:
text_preprocessingdf.head()

,title,director,cast,description
0,3%,NaN,"JoÃ£o Miguel, Bianca Comparato, Michel Gomes, ...",In a future where the elite inhabit an island ...
1,7:19,Jorge Michel Grau,"DemiÃ¡n Bichir, HÃ©ctor Bonilla, Oscar Serrano...",After a devastating earthquake hits Mexico Cit...
2,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...","When an army recruit is found dead, his fellow..."
3,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...","In a postapocalyptic world, rag-doll robots hi..."
4,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",A brilliant group of students become card-coun...


In [ ]:
text_preprocessingdf.shape

(7777, 4)

In [ ]:
text_preprocessingdf.isnull().sum()

,0
title,0
director,2379
cast,718
description,0


In [ ]:
text_preprocessingdf = text_preprocessingdf.fillna('')

In [ ]:
def fix_mojibake(text):
    try:
        return text.encode('latin1').decode('utf-8')
    except:
        return text

In [ ]:
#Applying encoding to columns to fix errors
for col in text_preprocessingdf.columns:
    text_preprocessingdf[col] = text_preprocessingdf[col].apply(fix_mojibake)

In [ ]:
text_preprocessingdf.head(10)

,title,director,cast,description
0,3%,,"João Miguel, Bianca Comparato, Michel Gomes, R...",In a future where the elite inhabit an island ...
1,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",After a devastating earthquake hits Mexico Cit...
2,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...","When an army recruit is found dead, his fellow..."
3,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...","In a postapocalyptic world, rag-doll robots hi..."
4,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",A brilliant group of students become card-coun...
5,46,Serdar Akar,"Erdal Beşikçioğlu, Yasemin Allen, Melis Birkan...",A genetics professor experiments with a treatm...
6,122,Yasir Al Yasiri,"Amina Khalil, Ahmed Dawood, Tarek Lotfy, Ahmed...","After an awful accident, a couple admitted to ..."
7,187,Kevin Reynolds,"Samuel L. Jackson, John Heard, Kelly Rowan, Cl...",After one of his high school students attacks ...
8,706,Shravan Kumar,"Divya Dutta, Atul Kulkarni, Mohan Agashe, Anup...","When a doctor goes missing, his psychiatrist w..."
9,1920,Vikram Bhatt,"Rajneesh Duggal, Adah Sharma, Indraneil Sengup...",An architect and his wife move into a castle t...


In [ ]:
#For detecting stopwords like the,to etc
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
def preprocess_description(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)  # remove html tags
    text = re.sub(r'http\S+|www\S+', ' ', text)  # remove urls
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)  # keep only alphabets
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces

    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return ' '.join(tokens)

In [ ]:
#Applying tokenization, lemmatization on description column
text_preprocessingdf['description_clean'] = text_preprocessingdf['description'].apply(preprocess_description)

#####Preprocessing title column

In [ ]:
def preprocess_title(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)  # keep alphabets and digits
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
text_preprocessingdf['title_clean'] = text_preprocessingdf['title'].apply(preprocess_title)

In [ ]:
text_preprocessingdf[text_preprocessingdf['title'].isin(['3%', '9', '7:19', '23:59', '21'])][['title', 'title_clean']]

,title,title_clean
0,3%,3
1,7:19,7 19
2,23:59,23 59
3,9,9
4,21,21


####Preprocessing director column

In [ ]:
def preprocess_director(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z,\s]', ' ', text)  # keep letters, commas, spaces
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.replace(',', ' ')
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
text_preprocessingdf['director_clean'] = text_preprocessingdf['director'].apply(preprocess_director)

####Preprocessing cast column

In [ ]:
def preprocess_cast(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z,\s]', ' ', text)  # keep letters, commas, spaces
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.replace(',', ' ')
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
text_preprocessingdf['cast_clean'] = text_preprocessingdf['cast'].apply(preprocess_cast)

In [ ]:
text_preprocessingdf[['title', 'title_clean',
                      'director', 'director_clean',
                      'cast', 'cast_clean',
                      'description', 'description_clean']].head(10)

,title,title_clean,director,director_clean,cast,cast_clean,description,description_clean
0,3%,3,,,"João Miguel, Bianca Comparato, Michel Gomes, R...",jo o miguel bianca comparato michel gomes rodo...,In a future where the elite inhabit an island ...,future elite inhabit island paradise far crowd...
1,7:19,7 19,Jorge Michel Grau,jorge michel grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",demi n bichir h ctor bonilla oscar serrano aza...,After a devastating earthquake hits Mexico Cit...,devastating earthquake hit mexico city trapped...
2,23:59,23 59,Gilbert Chan,gilbert chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",tedd chan stella chung henley hii lawrence koh...,"When an army recruit is found dead, his fellow...",army recruit found dead fellow soldier forced ...
3,9,9,Shane Acker,shane acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",elijah wood john c reilly jennifer connelly ch...,"In a postapocalyptic world, rag-doll robots hi...",postapocalyptic world rag doll robot hide fear...
4,21,21,Robert Luketic,robert luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",jim sturgess kevin spacey kate bosworth aaron ...,A brilliant group of students become card-coun...,brilliant group student become card counting e...
5,46,46,Serdar Akar,serdar akar,"Erdal Beşikçioğlu, Yasemin Allen, Melis Birkan...",erdal be ik io lu yasemin allen melis birkan s...,A genetics professor experiments with a treatm...,genetics professor experiment treatment comato...
6,122,122,Yasir Al Yasiri,yasir al yasiri,"Amina Khalil, Ahmed Dawood, Tarek Lotfy, Ahmed...",amina khalil ahmed dawood tarek lotfy ahmed el...,"After an awful accident, a couple admitted to ...",awful accident couple admitted grisly hospital...
7,187,187,Kevin Reynolds,kevin reynolds,"Samuel L. Jackson, John Heard, Kelly Rowan, Cl...",samuel l jackson john heard kelly rowan clifto...,After one of his high school students attacks ...,one high school student attack dedicated teach...
8,706,706,Shravan Kumar,shravan kumar,"Divya Dutta, Atul Kulkarni, Mohan Agashe, Anup...",divya dutta atul kulkarni mohan agashe anupam ...,"When a doctor goes missing, his psychiatrist w...",doctor go missing psychiatrist wife treat biza...
9,1920,1920,Vikram Bhatt,vikram bhatt,"Rajneesh Duggal, Adah Sharma, Indraneil Sengup...",rajneesh duggal adah sharma indraneil sengupta...,An architect and his wife move into a castle t...,architect wife move castle slated become luxur...


In [ ]:
nlp_clean_df = text_preprocessingdf[[
    'title_clean',
    'director_clean',
    'cast_clean',
    'description_clean'
]].copy()

In [ ]:
nlp_clean_df.head()

,title_clean,director_clean,cast_clean,description_clean
0,3,,jo o miguel bianca comparato michel gomes rodo...,future elite inhabit island paradise far crowd...
1,7 19,jorge michel grau,demi n bichir h ctor bonilla oscar serrano aza...,devastating earthquake hit mexico city trapped...
2,23 59,gilbert chan,tedd chan stella chung henley hii lawrence koh...,army recruit found dead fellow soldier forced ...
3,9,shane acker,elijah wood john c reilly jennifer connelly ch...,postapocalyptic world rag doll robot hide fear...
4,21,robert luketic,jim sturgess kevin spacey kate bosworth aaron ...,brilliant group student become card counting e...


#####Vectorizing text columns

I used TF-IDF for semantically rich text fields like description and title, and CountVectorizer for entity-based fields like cast and director, where the presence of names was more informative than relative term weighting.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [ ]:
description_vectorizer = TfidfVectorizer(
    max_features=1500,
    ngram_range=(1, 2),
    min_df=3
)

description_matrix = description_vectorizer.fit_transform(nlp_clean_df['description_clean'])

In [ ]:
description_matrix.shape

(7777, 1500)

In [ ]:
title_vectorizer = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    min_df=2
)

title_matrix = title_vectorizer.fit_transform(nlp_clean_df['title_clean'])
title_matrix.shape

(7777, 300)

In [ ]:
cast_vectorizer = CountVectorizer(
    max_features=500,
    min_df=2
)

cast_matrix = cast_vectorizer.fit_transform(nlp_clean_df['cast_clean'])
cast_matrix.shape

(7777, 500)

In [ ]:
director_vectorizer = CountVectorizer(
    max_features=150,
    min_df=2
)

director_matrix = director_vectorizer.fit_transform(nlp_clean_df['director_clean'])
director_matrix.shape

(7777, 150)

In [ ]:
description_features = description_vectorizer.get_feature_names_out()
description_features[:30]

array(['abandoned', 'ability', 'abuse', 'academy', 'accident',
       'accidentally', 'account', 'accused', 'across', 'act', 'action',
       'activist', 'actor', 'actress', 'adaptation', 'addiction', 'adult',
       'adventure', 'affair', 'africa', 'african', 'african american',
       'aftermath', 'age', 'aged', 'agency', 'agent', 'aging', 'ago',
       'agrees'], dtype=object)

In [ ]:
title_features = title_vectorizer.get_feature_names_out()
title_features[:30]

array(['100', 'about', 'adventure', 'adventures', 'after', 'again', 'age',
       'all', 'all the', 'am', 'america', 'american', 'an', 'and',
       'and the', 'are', 'art', 'at', 'at the', 'aur', 'baby', 'back',
       'bad', 'barbie', 'battle', 'be', 'before', 'beginning', 'behind',
       'best'], dtype=object)

In [ ]:
cast_features = cast_vectorizer.get_feature_names_out()
cast_features[:30]

array(['aaron', 'abdel', 'adam', 'adams', 'adrian', 'ahmed', 'ai', 'ajay',
       'akshay', 'al', 'alan', 'alejandro', 'alessandro', 'alex',
       'alexander', 'ali', 'alice', 'allen', 'amanda', 'amber', 'amy',
       'an', 'ana', 'anand', 'anderson', 'andr', 'andrea', 'andrew',
       'andy', 'angela'], dtype=object)

In [ ]:
director_features = director_vectorizer.get_feature_names_out()
director_features[:30]

array(['aaron', 'abhishek', 'adam', 'ahmad', 'ahmed', 'akhtar', 'al',
       'alejandro', 'alex', 'ali', 'alibhai', 'an', 'anderson', 'andrew',
       'andy', 'anthony', 'antonio', 'anurag', 'bell', 'ben', 'brian',
       'campos', 'carlos', 'cathy', 'chahine', 'chapman', 'charles',
       'chris', 'christian', 'christopher'], dtype=object)

In [ ]:
print("Description matrix shape:", description_matrix.shape)
print("Title matrix shape:", title_matrix.shape)
print("Cast matrix shape:", cast_matrix.shape)
print("Director matrix shape:", director_matrix.shape)

Description matrix shape: (7777, 1500)
Title matrix shape: (7777, 300)
Cast matrix shape: (7777, 500)
Director matrix shape: (7777, 150)
